# Construction du dataset final

Merge de toutes les sources :
- `rider_data/` : résultats + features course (déjà joinés par le collègue) + `leader_played`
- `race_data/max_pts_per_race.csv` : points max possible par course
- `race_data/surface_sections.csv` : cobblestone/gravel dans les derniers 10/20/50 km
- Calcul de la **forme** (rolling pts_uci par coureur)
- Calcul du **pts_ratio** = pts_uci / max_pts

**Output** : `race_data/final_dataset.csv`

In [16]:
import os, glob
import numpy as np
import pandas as pd
from tqdm import tqdm

RIDER_DIR   = 'rider_data/'
MAX_PTS_CSV = 'race_data/max_pts_per_race.csv'
SURF_CSV    = 'race_data/surface_sections.csv'
MATCH_CSV   = 'data/matching/all/dataset_final.csv'
OUTPUT_CSV  = 'race_data/final_dataset.csv'

In [17]:
# ── 1. Charger tous les rider CSVs ────────────────────────────────────────────
# ~20 000 fichiers, ~4.6M lignes

files = glob.glob(os.path.join(RIDER_DIR, '*.csv'))
print(f'Fichiers rider : {len(files)}')

chunks = []
for f in tqdm(files, desc='Chargement rider_data'):
    try:
        df = pd.read_csv(f, low_memory=False)
        rider_name = os.path.splitext(os.path.basename(f))[0]
        df['rider'] = rider_name
        chunks.append(df)
    except Exception:
        pass

df_all = pd.concat(chunks, ignore_index=True)
df_all['date'] = pd.to_datetime(df_all['date'], errors='coerce')

print(f'Shape total : {df_all.shape}')
print(f'Colonnes : {list(df_all.columns)}')

Fichiers rider : 20049


Chargement rider_data: 100%|██████████| 20049/20049 [01:48<00:00, 185.42it/s]
/var/folders/w5/_dp1f8855qd_31v9b5ywhggh0000gn/T/ipykernel_69084/3810550122.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat(chunks, ignore_index=True)


Shape total : (3262766, 60)
Colonnes : ['date', 'year', 'course', 'type', 'etape', 'equipe', 'selected', 'classification', 'statut', 'rang', 'pts_uci', 'rang_gc_final', 'pts_uci_gc_final', 'rang_kom_final', 'pts_uci_kom_final', 'rang_points_final', 'pts_uci_points_final', 'pts_uci_equipe_stage', 'pts_uci_equipe_total', 'pts_uci_gc', 'pts_uci_kom', 'pts_uci_points', 'stage_num', 'distance_gpx_km', 'asphalt_km', 'paved_km', 'cobblestones_km', 'unpaved_km', 'compacted_gravel_km', 'road_km', 'street_km', 'state_road_km', 'cycleway_km', 'off-grid_(unknown)_km', 'access_road_km', 'unknown_km', 'path_km', 'singletrack_km', 'denivele_pos', 'denivele_neg', 'altitude_max', 'altitude_min', 'n_cols_cat4', 'n_cols_cat3', 'n_cols_cat2', 'n_cols_cat1', 'n_cols_hc', 'loc_last_col_cat4', 'loc_last_col_cat3', 'loc_last_col_cat2', 'loc_last_col_cat1', 'loc_last_col_hc', 'gradient_last_1km', 'gradient_last_3km', 'gradient_last_5km', 'denivele_last_5km', 'gradient_first_50km', 'denivele_first_50km', 'leade

In [18]:
# ── 2. Construire la clé stage pour le join max_pts ──────────────────────────
# rider_data: type='stage' + stage_num=3 → 'stage_3'
# rider_data: type='result'              → 'result'
# rider_data: type='prologue'            → 'prologue'

def build_stage_key(row):
    t = str(row['type']).lower()
    if t == 'stage':
        sn = row.get('stage_num')
        if pd.notna(sn):
            return f'stage_{int(sn)}'
        # fallback : etape
        et = row.get('etape')
        if pd.notna(et):
            return f'stage_{int(et)}'
        return 'stage'
    return t

df_all['stage_key'] = df_all.apply(build_stage_key, axis=1)
print('stage_key exemples :', df_all['stage_key'].value_counts().head(8).to_dict())

stage_key exemples : {'result': 896288, 'stage_2': 399515, 'stage_1': 386671, 'stage_3': 376935, 'stage_4': 332002, 'stage_5': 250718, 'stage_6': 133423, 'stage_7': 101633}


In [ ]:
# ── 3. Calculer max_pts et pts_ratio depuis le rider_data ─────────────────────
# max_pts = max(pts_uci) par course/year/stage_key (même échelle que pts_uci)
# pts_ratio = NaN si max_pts == 0 (pas de points UCI alloués pour cette course)

max_from_data = df_all.groupby(['course', 'year', 'stage_key'])['pts_uci'].max().reset_index(name='max_pts')
df_all = df_all.merge(max_from_data, on=['course', 'year', 'stage_key'], how='left')
df_all['pts_ratio'] = (df_all['pts_uci'] / df_all['max_pts']).where(df_all['max_pts'] > 0).clip(0, 1)

winners = df_all[df_all['rang'] == 1]
exact_1 = (winners['pts_ratio'] == 1.0).sum()
print(f'max_pts NaN : {df_all["max_pts"].isna().sum()}')
print(f'Vainqueurs rang=1 avec pts_ratio==1.0 : {exact_1}/{len(winners)} ({exact_1/len(winners)*100:.1f}%)')

In [20]:
# ── 4. Joindre surface_sections via fichier_gpx ───────────────────────────────

df_match = pd.read_csv(MATCH_CSV)
df_match['stage_key'] = df_match['stage'].str.lower()
df_match = df_match.rename(columns={'nom_pcs': 'course', 'annee': 'year'})
gpx_lookup = df_match[['course', 'year', 'stage_key', 'fichier_gpx']].drop_duplicates()

df_surf = pd.read_csv(SURF_CSV)
df_surf['fichier_gpx'] = df_surf['fichier_gpx'].str.replace(' / ', ' - ', regex=False)

df_all = df_all.merge(gpx_lookup, on=['course', 'year', 'stage_key'], how='left')
df_all = df_all.merge(df_surf, on='fichier_gpx', how='left')

# Les courses sans cobblestone/gravel significatif ne sont pas dans surface_sections
# → NaN signifie 0 km (pas de pavés), pas une donnée manquante
surf_cols = [c for c in df_surf.columns if c != 'fichier_gpx']
df_all[surf_cols] = df_all[surf_cols].fillna(0)

print(f'Colonnes surfaces : {surf_cols}')
print(f'Couverture (non-zéro) : {(df_all[surf_cols[0]] > 0).mean()*100:.1f}% ont du cobblestone')

Colonnes surfaces : ['cobblestones_last_50km', 'compacted_gravel_last_50km', 'cobblestones_last_20km', 'compacted_gravel_last_20km', 'cobblestones_last_10km', 'compacted_gravel_last_10km']
Couverture (non-zéro) : 17.9% ont du cobblestone


In [ ]:
# ── 5. Colonnes finales et export ─────────────────────────────────────────────

IDENTIF  = ['rider', 'date', 'year', 'course', 'stage_key', 'stage_num',
             'equipe', 'classification', 'statut', 'type', 'fichier_gpx']

TARGET   = ['selected', 'rang', 'pts_uci', 'pts_ratio',
             'pts_uci_equipe_stage', 'pts_uci_equipe_total',
             'pts_uci_gc', 'pts_uci_kom', 'pts_uci_points',
             'rang_gc_final', 'pts_uci_gc_final',
             'rang_kom_final', 'pts_uci_kom_final',
             'rang_points_final', 'pts_uci_points_final',
             'max_pts', 'leader_played']

RACE     = ['distance_gpx_km', 'asphalt_km', 'paved_km', 'cobblestones_km',
             'unpaved_km', 'compacted_gravel_km', 'road_km', 'street_km',
             'state_road_km', 'cycleway_km', 'path_km', 'singletrack_km',
             'denivele_pos', 'denivele_neg', 'altitude_max', 'altitude_min',
             'n_cols_cat4', 'n_cols_cat3', 'n_cols_cat2', 'n_cols_cat1', 'n_cols_hc',
             'loc_last_col_cat4', 'loc_last_col_cat3', 'loc_last_col_cat2',
             'loc_last_col_cat1', 'loc_last_col_hc',
             'gradient_last_1km', 'gradient_last_3km', 'gradient_last_5km',
             'denivele_last_5km', 'gradient_first_50km', 'denivele_first_50km']

SURFACE  = ['cobblestones_last_50km', 'compacted_gravel_last_50km',
             'cobblestones_last_20km', 'compacted_gravel_last_20km',
             'cobblestones_last_10km', 'compacted_gravel_last_10km']

LOAD     = ['n_races_30d', 'km_30d']

keep = [c for c in IDENTIF + TARGET + RACE + SURFACE + LOAD if c in df_all.columns]
df_final = df_all[keep].copy()

print(f'Shape final : {df_final.shape}')
print(f'Colonnes ({len(df_final.columns)}) : {list(df_final.columns)}')

df_final.to_csv(OUTPUT_CSV, index=False)
print(f'\nSauvegardé : {OUTPUT_CSV}')

In [22]:
# ── 6. Validation rapide ──────────────────────────────────────────────────────

df = pd.read_csv(OUTPUT_CSV)
print('Shape :', df.shape)
print()

# Couverture des nouvelles features
new_feats = ['max_pts', 'pts_ratio', 'leader_played'] + SURFACE
for c in [f for f in new_feats if f in df.columns]:
    pct = df[c].notna().mean() * 100
    print(f'  {c:45s}  {pct:5.1f}% non-NaN')

print()
print('pts_ratio (coureurs sélectionnés, selected=1) :')
sel = df[df['selected'] == 1]
print(sel['pts_ratio'].describe())

print()
print('leader_played (selected=1) :')
print(sel['leader_played'].value_counts())

Shape : (3262766, 75)

  max_pts                                        100.0% non-NaN
  pts_ratio                                      100.0% non-NaN
  leader_played                                   97.1% non-NaN
  loc_last_col_cat4                               91.7% non-NaN
  loc_last_col_cat3                               91.7% non-NaN
  loc_last_col_cat2                               91.7% non-NaN
  loc_last_col_cat1                               91.7% non-NaN
  loc_last_col_hc                                 91.7% non-NaN
  gradient_last_1km                               91.7% non-NaN
  gradient_last_3km                               91.7% non-NaN
  gradient_last_5km                               91.7% non-NaN
  denivele_last_5km                               91.7% non-NaN
  cobblestones_last_50km                         100.0% non-NaN
  compacted_gravel_last_50km                     100.0% non-NaN
  cobblestones_last_20km                         100.0% non-NaN
  compacted_grave